# MovieLens 电影标题 embedding：手工调试

本 Notebook 保持未执行状态。请自己从上到下运行单元格，观察类型、形状和内容，并在 Markdown 中记录你的理解。先检查数据集前两部电影，再分批导出整个数据集。

`mode="sentence"` 经过完整 BERT；`mode="word"` 查询输入词向量表。标题均从 `data/ml-100k/u.item` 读取。

## 1. 导入与工作目录

请从项目根目录启动 Jupyter，并选择 **NCF-MF (.venv)** 内核。

In [ ]:
import csv
from pathlib import Path

import torch
from transformers import AutoModel, AutoTokenizer

In [ ]:
import sys
sys.executable

In [ ]:
project_dir = Path.cwd()
project_dir

In [ ]:
data_path = project_dir / "data" / "ml-100k" / "u.item"
data_path.exists()

上一步应找到文件。如果没有，请自己将 `project_dir` 改成项目的绝对路径。Notebook 中不使用 `__file__`。

## 2. 读取数据集

观察原始记录的第 1、2 列，以及字符串 ID 转为整数后的变化。

In [ ]:
with data_path.open("r", encoding="latin-1", newline="") as file:
    reader = csv.reader(file, delimiter="|", quoting=csv.QUOTE_NONE)
    records = list(reader)

In [ ]:
len(records), records[0][:2]

In [ ]:
records.sort(key=lambda row: int(row[0]))

In [ ]:
movie_ids = [int(row[0]) for row in records]

In [ ]:
titles = [row[1] for row in records]

In [ ]:
type(titles), len(titles), movie_ids[:3], titles[:3]

## 3. 配置并加载模型

默认使用本机已经缓存的 `bert-base-uncased`。换电脑时可将 `model_name` 改成本地模型目录；或者在联网条件下把 `local_files_only` 改为 `False`。

In [ ]:
mode = "sentence"
model_name = "bert-base-uncased"
batch_size = 32

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name, local_files_only=True)

In [ ]:
model = AutoModel.from_pretrained(model_name, local_files_only=True, add_pooling_layer=False)

In [ ]:
model.eval()
model.requires_grad_(False)
model.config.hidden_size

## 4. 手工检查前两部电影的分词

先逐步运行这一小批输入。观察补齐后的长度与原始标题长度的区别。

In [ ]:
batch_titles = titles[:2]
batch_titles

In [ ]:
inputs = tokenizer(
    batch_titles,
    padding=True,
    truncation=True,
    max_length=model.config.max_position_embeddings,
    return_special_tokens_mask=True,
    return_tensors="pt",
)

In [ ]:
type(inputs), list(inputs.keys())

In [ ]:
input_ids = inputs["input_ids"]
input_ids

In [ ]:
input_ids.shape, input_ids.dtype

In [ ]:
tokens = tokenizer.convert_ids_to_tokens(input_ids[0].tolist())
tokens

In [ ]:
attention_mask = inputs["attention_mask"]
attention_mask

In [ ]:
special_tokens_mask = inputs.pop("special_tokens_mask")
special_tokens_mask

## 5. 获得 token 向量

`if/else` 代码块需要完整放在同一个单元格。模型已冻结，因此可以逐步查看下面的普通张量。需要跟踪块内的行时，可使用你自己的 Notebook 调试器。

In [ ]:
if mode == "word":
    word_embedding_layer = model.get_input_embeddings()
    hidden = word_embedding_layer(input_ids)
else:
    outputs = model(**inputs)
    hidden = outputs.last_hidden_state

In [ ]:
type(hidden), hidden.shape, hidden.requires_grad

## 6. 排除特殊符号和 padding，手工检查均值池化

依次观察 `[B, L] → [B, L, 1]` 的广播，以及 `[B, L, H] → [B, H]` 的变化。

In [ ]:
valid_mask = attention_mask.bool() & ~special_tokens_mask.bool()
valid_mask

In [ ]:
expanded_mask = valid_mask.unsqueeze(-1)
expanded_mask.shape

In [ ]:
masked_hidden = hidden * expanded_mask
masked_hidden.shape

In [ ]:
embedding_sum = masked_hidden.sum(dim=1)
embedding_sum.shape

In [ ]:
token_counts = valid_mask.sum(dim=1, keepdim=True).clamp_min(1)
token_counts

In [ ]:
embeddings = embedding_sum / token_counts
embeddings.shape

In [ ]:
embeddings[0, :8]

### 我的手工观察

请自己填写：输入与输出的类型、每一步的维度、mask 的作用、两种 mode 的区别，以及还有哪些不理解的地方。

## 7. 分批处理全部电影

完成前面的小批手工观察后，再运行这个完整循环。其计算与独立脚本相同，`for/with` 块需要放在一个单元格中。这个单元格目前未执行。

In [ ]:
# 3. 分批处理所有电影，避免一次将整个数据集送入 BERT。
embedding_batches = []
with torch.no_grad():
    for start in range(0, len(titles), batch_size):
        batch_titles = titles[start:start + batch_size]
        inputs = tokenizer(
            batch_titles,
            padding=True,
            truncation=True,
            max_length=model.config.max_position_embeddings,
            return_special_tokens_mask=True,
            return_tensors="pt",
        )
        input_ids = inputs["input_ids"]
        attention_mask = inputs["attention_mask"]
        special_tokens_mask = inputs.pop("special_tokens_mask")
        tokens = tokenizer.convert_ids_to_tokens(input_ids[0].tolist())

        # 4. 得到 [批次电影数, 序列长度, 768] 的 token 向量。
        if mode == "word":
            word_embedding_layer = model.get_input_embeddings()
            hidden = word_embedding_layer(input_ids)
        else:
            outputs = model(**inputs)
            hidden = outputs.last_hidden_state

        # 5. 排除自动加入的 [CLS]、[SEP] 和 padding，再取均值。
        valid_mask = attention_mask.bool() & ~special_tokens_mask.bool()
        expanded_mask = valid_mask.unsqueeze(-1)
        masked_hidden = hidden * expanded_mask
        embedding_sum = masked_hidden.sum(dim=1)
        token_counts = valid_mask.sum(dim=1, keepdim=True).clamp_min(1)
        embeddings = embedding_sum / token_counts
        embedding_batches.append(embeddings)

## 8. 导出并读取检查

第 i 行向量必须对应 `movie_ids[i]` 和 `titles[i]`。Notebook 的导出文件带 `_notebook` 后缀，与独立脚本的运行验证文件区分。

In [ ]:
movie_embeddings = torch.cat(embedding_batches, dim=0)
movie_embeddings.shape

In [ ]:
output_path = project_dir / "outputs" / f"movie_titles_{mode}_notebook.pt"
output_path.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
torch.save({"movie_ids": movie_ids, "titles": titles, "embeddings": movie_embeddings, "mode": mode}, output_path)
output_path

In [ ]:
saved = torch.load(output_path, weights_only=True)
saved["embeddings"].shape

In [ ]:
saved["movie_ids"][0], saved["titles"][0], saved["embeddings"][0, :8]

### 衍生用法：手工比较

1. 记录当前输出，再将 `mode` 改为 `word` 并从配置单元格重新运行。
2. 改变前面 `titles[:2]` 中的数量，观察 padding 和张量形状。
3. 根据电影 ID 找到对应向量，例如用 `saved["movie_ids"].index(电影ID)` 得到行号。
4. 需要排除乱序执行的影响时，重启 Kernel，然后从上往下手工运行。